<a href="https://colab.research.google.com/github/hanghae-plus-AI/AI-1-gwkcareer/blob/main/week5/Chapter3_2_%EC%8B%AC%ED%99%94%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [5주차 심화 과제] 수능 국어 문제 GPT-4로 풀어보기

이번 과제에서는 2023년도 수능 국어 문제를 GPT-4로 풀어볼 것입니다. 아래 요구사항들을 지켜주시면 됩니다.

- 수능 국어 문제를 준비합니다. 다음 github의 `data > 2023_11_KICE.json` data를 colab으로 불러오시면 됩니다:
    
    [GitHub - NomaDamas/KICE_slayer_AI_Korean: 수능 국어 1등급에 도전하는 AI](https://github.com/NomaDamas/KICE_slayer_AI_Korean)
    
- 하나의 문제에 대해서 GPT-4의 예측 결과를 내놓는 함수를 `def prediction(problem)`이라는 signature로 만드셔야 합니다. `problem`은 json 형태의 문제입니다. 내부는 logit 계산을 통해 구현하거나 순수하게 text 생성으로 해결하셔도 좋습니다. ***단, 2023년도 수능 국어의 정답을 활용하시면 안됩니다.***
- `def prediction` 함수를 모든 수능 국어 문제들에 대해서 돌린 후, 실제 정답과 비교하여 GPT-4의 점수를 계산하는 코드를 구현하시면 됩니다. ***단, 점수 계산은 모두 코드를 통해서만 진행되어야 합니다.*** 사람이 직접 GPT-4의 출력 결과를 보고 대조하는 형식으로 되면 안됩니다.
- 채점 결과 50점을 넘기면 통과입니다.

이외에는 제약사항이 없습니다. 위의 제약사항만 만족한다면 GPT-4 이외의 open LLM을 사용하셔도 좋습니다.

In [1]:
!pip install --upgrade openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.9/386.9 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.2/325.2 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 4.9 MB/s eta 0:00:00


In [15]:
import pandas as pd

# raw 파일 URL 사용
url = "https://raw.githubusercontent.com/NomaDamas/KICE_slayer_AI_Korean/master/data/2023_11_KICE.json"
data = pd.read_json(url)

# 데이터 확인
print(data.info())
print(data['paragraph'])  ## 지문
print(data['problems']) ## 문제/선택지/답/점

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         11 non-null     object
 1   paragraph  11 non-null     object
 2   type       11 non-null     int64 
 3   problems   11 non-null     object
dtypes: int64(1), object(3)
memory usage: 480.0+ bytes
None
0     사람들이 지속적으로 책을 읽는 이유 중 하나는 즐거움이다. 독서의 즐거움에는 여러 ...
1     (가)[A](중국에서 비롯된 유서(類書)는 고금의 서적에서 자료를 수집하고 항목별로...
2     법령의 조문은 대개 ‘A에 해당하면 B를 해야 한다.’처럼 요건과효과로 구성된 조건...
3     하루에 필요한 에너지의 양은 하루 동안의 총 열량 소모량인 대사량으로 구한다. 그중...
4     혼례를 마친 후 최척이 아내와 함께 장모를 모시고 집으로 돌아오매 하인들이 기뻐했다...
5     (가)[A](이런들 어떠하며 저런들 어떠하료초야우생(草野愚生)이 이렇다 어떠하료하물...
6     밤이 깊어지면, 시장 안의 가게들은 하나씩 문을 닫고, 길가에 리어카를 놓고 팔던 ...
7     (가) 한여름 채전으로 ㉠(가 보아라) 수염을 드리운 몇 그루 옥수수에 가지, 고추...
8     안녕하세요? 발표를 맡은 ○○○입니다. 지난 수업 시간에 우리는 도로에서 볼 수 있...
9     (가)는 ○○고등학교 행사에 참여한 학생이 마을 소식지에 쓴 후기이고, (나)는 이...
10    우리나라의 연간 1인당 커피 소비량은 세계 평균의 2배 이상일 정도로



```
{
    'question': '윗글의 내용과 일치하지 않는 것은?',
    'choices': [
        '같은 책을 읽은 독자라도 서로 다른 의미를 구성할 수 있다.',
        '다른 독자와의 소통은 독자가 인식의 폭을 확장하도록 돕는다',
        '독자는 직접 경험해 보지 못했던 다양한 삶을 책의 필자를 매개로 접할 수 있다.',
        '독자의 배경지식, 관점, 읽기 환경, 과제는 독자의 의미 구성에 영향을 주는 독자 요인이다.',
        '독자는 책을 읽을 때 자신이 속한 사회나 시대의 영향을 받으며 필자와 간접적으로 대화한다'
    ],
    'answer': 4,  # 정답은 네 번째 선택지
    'score': 2    # 배점
}
```



In [21]:
from openai import OpenAI
from google.colab import userdata
import re

# OpenAI API 키 설정
GPTclient = OpenAI(api_key=userdata.get('opanai_key'))

def prediction(problem, paragraph):
    question = problem['question']  # 문제
    choices = problem['choices']    # 선택지

    # GPT-4로 질문에 대한 응답 생성
    prompt = f"지문: {paragraph}\n\n문제: {question}\n선택지:\n"
    for i, choice in enumerate(choices):
        prompt += f"{i+1}. {choice}\n"

    prompt += "\n정답을 고르시오."

    response = GPTclient.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ]
    )

    # GPT-4의 답변에서 선택된 답 추출
    gpt_response = response.choices[0].message.content

    # 정규 표현식으로 숫자 추출 (답이 숫자 형식으로 나올 경우를 가정)
    predicted_answer = None
    match = re.search(r'\b\d+\b', gpt_response)
    if match:
        predicted_answer = int(match.group())

    # 예외 처리: 답을 추출하지 못하면 1번 선택
    if predicted_answer is None:
        predicted_answer = 1

    return predicted_answer

In [ ]:
# 채점 및 결과 계산
correct_score_sum = 0

for i, row in data.iterrows():
    paragraph = row['paragraph']  # 지문
    problems_list = row['problems']  # 문제 리스트

    for problem in problems_list:
        predicted_answer = prediction(problem, paragraph)  # 예측된 답

        # 실제 정답과 비교하여 맞힌 문제의 배점 합산
        if predicted_answer == problem['answer']:
            correct_score_sum += problem['score']  # 맞힌 문제의 배점 합산

# 최종 점수 출력
print(f"GPT-4의 최종 점수: {correct_score_sum}")